# 11 - One-Hot to Embedding

Goal: Compress one-hot encoding (10 words → 3D embeddings)

Run with: conda activate tfenv

In [9]:
# Celda 1: Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import plotly.express as px

In [10]:
# Celda 2: Vocabulary
vocab = ["el", "la", "gato", "perro", "come", "duerme", "en", "casa", "corre", "nada"]  # 10 unique words
vocab_size = len(vocab)
embed_dim = 3  # 3D for visualization

print(f"Vocab size: {vocab_size}")
print(f"Words: {vocab}")

# One-hot encoding
def one_hot(word_idx, vocab_size):
    vector = np.zeros(vocab_size)
    vector[word_idx] = 1
    return vector

# Test one-hot
print(f"\nOne-hot for 'gato' (idx=2):")
print(one_hot(2, vocab_size))

Vocab size: 10
Words: ['el', 'la', 'gato', 'perro', 'come', 'duerme', 'en', 'casa', 'corre', 'nada']

One-hot for 'gato' (idx=2):
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]


In [11]:
# Celda 3: Method 1 - nn.Linear (manual compression)
print("=== Method 1: nn.Linear ===")
linear_layer = nn.Linear(vocab_size, embed_dim, bias=False)

# Compress one-hot to embedding
gato_onehot = torch.tensor(one_hot(2, vocab_size), dtype=torch.float32)
gato_embedding = linear_layer(gato_onehot)
print(f"Embedding for 'gato': {gato_embedding.detach().numpy()}")

=== Method 1: nn.Linear ===
Embedding for 'gato': [-0.11978603  0.02619928  0.3045832 ]


In [12]:
# Celda 4: Method 2 - nn.Embedding (built-in)
print("=== Method 2: nn.Embedding ===")
embedding_layer = nn.Embedding(vocab_size, embed_dim)

# Embedding layer expects index, not one-hot
gato_idx = torch.tensor([2])
gato_embedding2 = embedding_layer(gato_idx)
print(f"Embedding for 'gato': {gato_embedding2.detach().numpy()[0]}")

# Compare weights
print(f"\nLinear weights shape: {linear_layer.weight.shape}")
print(f"Embedding weights shape: {embedding_layer.weight.shape}")
print("\nBoth are the same concept - just different interfaces!")

=== Method 2: nn.Embedding ===
Embedding for 'gato': [-0.8391481  -1.2194817   0.46364102]

Linear weights shape: torch.Size([3, 10])
Embedding weights shape: torch.Size([10, 3])

Both are the same concept - just different interfaces!


In [13]:
# Celda 5: Visualize initial embeddings (3D interactive!)
print("=== Initial Embeddings (random) ===")

# Get all embeddings
embeddings = embedding_layer.weight.detach().numpy()

# Create DataFrame for Plotly
df = pd.DataFrame(embeddings, columns=['x', 'y', 'z'])
df['word'] = vocab
df['index'] = range(len(vocab))

# 3D scatter plot
fig = px.scatter_3d(df, x='x', y='y', z='z', text='word', 
                    title='Word Embeddings (Random Initialization)')
fig.update_traces(marker=dict(size=8), textposition='top center')
fig.show()

=== Initial Embeddings (random) ===


In [14]:
# Celda 6: Train embeddings
print("=== Training embeddings ===")

# Create training pairs (fake context)
pairs = [
    (0, 2),   # el-gato
    (0, 3),   # el-perro
    (2, 3),   # gato-perro (related animals)
    (2, 8),   # gato-corre (cats can run)
    (3, 8),   # perro-corre (dogs can run)
    (3, 9),   # perro-nada (dogs can swim)
    (4, 5),   # come-duerme (related actions)
    (6, 7),   # en-casa (preposition-place)
]

# Train embeddings to be close for related words
optimizer = optim.Adam(embedding_layer.parameters(), lr=0.1)
criterion = nn.MSELoss()

for epoch in range(500):
    total_loss = 0
    for word_idx, context_idx in pairs:
        word = torch.tensor([word_idx])
        context = torch.tensor([context_idx])

        word_emb = embedding_layer(word)
        context_emb = embedding_layer(context)

        # Try to make related words have similar embeddings
        loss = criterion(word_emb, context_emb * 0.5)
        total_loss += loss

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.item():.4f}")

=== Training embeddings ===
Epoch 0, Loss: 10.0293
Epoch 100, Loss: 0.0029
Epoch 200, Loss: 0.0004
Epoch 300, Loss: 0.0000
Epoch 400, Loss: 0.0000


In [15]:
# Celda 7: Visualize trained embeddings (3D interactive!)
print("=== Trained Embeddings ===")

# Get trained embeddings
embeddings_trained = embedding_layer.weight.detach().numpy()

# Create DataFrame for Plotly
df_trained = pd.DataFrame(embeddings_trained, columns=['x', 'y', 'z'])
df_trained['word'] = vocab

# 3D scatter plot
fig = px.scatter_3d(df_trained, x='x', y='y', z='z', text='word',
                    title='Word Embeddings (After Training)')
fig.update_traces(marker=dict(size=8, color='red'), textposition='top center')
fig.show()

print("\n💡 Rotate the graph with your mouse to explore in 3D!")

=== Trained Embeddings ===



💡 Rotate the graph with your mouse to explore in 3D!


In [16]:
# Celda 8: Compare before and after
print("=== Comparison ===")
print("Before (random):")
print(embeddings[:5])
print("\nAfter (trained):")
print(embeddings_trained[:5])

=== Comparison ===
Before (random):
[[ 1.4292165e-03 -2.6174230e-04 -1.7914065e-06]
 [ 3.7476562e-02  4.7839004e-01  1.1286504e+00]
 [ 2.3800631e-03 -4.1971004e-04 -2.9630144e-06]
 [ 3.3204306e-03 -5.9265742e-04 -4.1510739e-06]
 [ 5.0857580e-01 -2.3333898e-01  1.7739791e-01]]

After (trained):
[[ 1.4292165e-03 -2.6174230e-04 -1.7914065e-06]
 [ 3.7476562e-02  4.7839004e-01  1.1286504e+00]
 [ 2.3800631e-03 -4.1971004e-04 -2.9630144e-06]
 [ 3.3204306e-03 -5.9265742e-04 -4.1510739e-06]
 [ 5.0857580e-01 -2.3333898e-01  1.7739791e-01]]
